# Challenge 2: Image Classification

## Module imports

In [1]:
import hparams
import image_classifier


import os
import torch
torch.manual_seed(hparams.SEED)
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision.transforms import v2 as transforms
from torch.utils.data import TensorDataset, DataLoader
from torchview import draw_graph
import cv2
import copy
import shutil
from itertools import product
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.gridspec as gridspec
import numpy as np


if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(hparams.SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


PyTorch version: 2.9.1+cpu
Device: cpu


## Image Loader

In [4]:
def load_images_from_folder(folder, cache_path):

    if os.path.exists(cache_path):
        print(f"Cache found at '{cache_path}'. Loading processed data...")
        return np.load(cache_path)

    print(f"Cache not found. Starting full image loading and preprocessing from '{folder}'...")
    
    images = []
    valid_extension = '.png'

    for filename in os.listdir(folder):
        if not filename.lower().endswith(valid_extension):
            continue

        file_path = os.path.join(folder, filename)
        img = cv2.imread(file_path)
        img = (img / 255.0).astype(np.float32)
        img = img[..., ::-1]
        dim = min(img.shape[:-1])
        h, w = img.shape[:2]
        img = img[(h - dim) // 2 : (h + dim) // 2, (w - dim) // 2 : (w + dim) // 2, :]
        img = cv2.resize(img, (hparams.IMAGE_SIZE, hparams.IMAGE_SIZE))
        images.append(img)
        
    final_array = np.array(images)

    cache_dir = os.path.dirname(cache_path)
    if cache_dir and not os.path.exists(cache_dir):
        os.makedirs(cache_dir, exist_ok=True)
        print(f"Created cache directory: {cache_dir}")
    
    print(f"Finished preprocessing {len(final_array)} images. Saving to cache...")
    np.save(cache_path, final_array)
    
    return final_array

train_images = load_images_from_folder(hparams.TRAIN_DATA_PATH, hparams.TRAIN_CACHE_PATH)
print(f"Loaded {len(train_images)} images for training")


Cache found at 'cache/train_images_cache.npy'. Loading processed data...
Loaded 2824 images for training
